In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import time
import numpy as np
from tqdm import tqdm
from torchmetrics.functional import calibration_error

In [3]:
# =========================================================
# DEVICE SETUP
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

✅ Using device: cuda


In [4]:
# =========================================================
# DATASET
# =========================================================
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:02<00:00, 66.0MB/s] 


In [5]:
# =========================================================
# MODEL SETUP (EfficientNet-B0)
# =========================================================
model = torchvision.models.efficientnet_b0(weights=None, num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

In [6]:
# =========================================================
# METRIC FUNCTIONS
# =========================================================
def topk_accuracy(output, target, topk=(1,)):
    maxk = max(topk)
    _, pred = output.topk(maxk, 1, True, True)
    correct = pred.eq(target.view(-1, 1).expand_as(pred))
    res = []
    for k in topk:
        correct_k = correct[:, :k].sum().item()
        res.append(correct_k)
    return res

def mean_off_diagonal_covariance(features, eps=1e-8):
    """Mean Off-diagonal Covariance (MOC) ↓"""
    X = features - features.mean(dim=0, keepdim=True)
    cov = (X.T @ X) / (X.shape[0] - 1)
    var = cov.diag()
    denom = torch.sqrt(var.unsqueeze(0) * var.unsqueeze(1)).clamp_min(eps)
    corr = cov / denom
    corr.fill_diagonal_(0)
    return corr.abs().mean().item()

def linear_CKA(X, Y):
    """Centered Kernel Alignment between two feature matrices"""
    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)
    hsic = torch.norm(X.T @ Y, p='fro') ** 2
    var1 = torch.norm(X.T @ X, p='fro') * torch.norm(Y.T @ Y, p='fro')
    return (hsic / var1).item()

In [7]:
# =========================================================
# TRAINING
# =========================================================
num_epochs = 100
best_acc = 0
best_val_loss = float('inf')
convergence_epoch = None
patience = 5
no_improve_epochs = 0
train_losses, test_losses = [], []
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss, total, correct_top1, correct_top5 = 0.0, 0, 0, 0
    epoch_start = time.time()

    for inputs, labels in tqdm(trainloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        top1, top5 = topk_accuracy(outputs, labels, topk=(1, 5))
        correct_top1 += top1
        correct_top5 += top5
        total += labels.size(0)

    train_loss = running_loss / len(trainloader)
    train_acc1 = 100 * correct_top1 / total
    train_acc5 = 100 * correct_top5 / total
    train_losses.append(train_loss)

    # =====================================================
    # VALIDATION
    # =====================================================
    model.eval()
    correct_top1, correct_top5, total = 0, 0, 0
    test_loss = 0.0
    probs, targets, features = [], [], []

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            top1, top5 = topk_accuracy(outputs, labels, topk=(1, 5))
            correct_top1 += top1
            correct_top5 += top5
            total += labels.size(0)

            probs.append(F.softmax(outputs, dim=1).cpu())
            targets.append(labels.cpu())

            # Use features before classifier for redundancy metrics
            feats = model.features(inputs).view(inputs.size(0), -1).cpu()
            features.append(feats)

    test_loss /= len(testloader)
    test_losses.append(test_loss)
    test_acc1 = 100 * correct_top1 / total
    test_acc5 = 100 * correct_top5 / total
    train_test_gap = abs(train_acc1 - test_acc1)

    all_probs = torch.cat(probs)
    all_targets = torch.cat(targets)
    ece = calibration_error(all_probs, all_targets, n_bins=15, task="multiclass", num_classes=10).item()

    all_features = torch.cat(features, dim=0)
    moc = mean_off_diagonal_covariance(all_features)

    epoch_time = time.time() - epoch_start

    print(f"\n📊 Epoch {epoch+1}:")
    print(f"Train Loss: {train_loss:.4f} | Train Acc@1: {train_acc1:.2f}% | Train Acc@5: {train_acc5:.2f}%")
    print(f"Val Loss: {test_loss:.4f} | Val Acc@1: {test_acc1:.2f}% | Val Acc@5: {test_acc5:.2f}%")
    print(f"Train–Test Gap: {train_test_gap:.2f}% | ECE: {ece:.4f} | MOC: {moc:.4f}")
    print(f"⏱ Epoch Time: {epoch_time:.2f}s")

    # Early stopping
    if test_loss < best_val_loss - 1e-4:
        best_val_loss = test_loss
        no_improve_epochs = 0
    else:
        no_improve_epochs += 1

    if no_improve_epochs >= patience:
        print(f"\n⏹ Early stopping triggered after {epoch+1} epochs (no improvement in {patience} epochs).")
        break

    if test_acc1 > best_acc:
        best_acc = test_acc1
        convergence_epoch = epoch + 1
        torch.save(model.state_dict(), "best_efficientnetb0_cifar10_mps.pth")

total_time = time.time() - start_time
print(f"\n✅ Training Complete.")
print(f"Best Top-1 Accuracy: {best_acc:.2f}% at Epoch {convergence_epoch}")
print(f"Total Training Time: {total_time/60:.2f} minutes")

# =========================================================
# INFERENCE TIME
# =========================================================
model.eval()
dummy_input = torch.randn(1, 3, 32, 32).to(device)
start = time.time()
_ = model(dummy_input)
inference_time = time.time() - start
print(f"\n⚡ Inference Time (1 sample): {inference_time*1000:.3f} ms")

Epoch 1/100: 100%|██████████| 391/391 [00:12<00:00, 30.51it/s]



📊 Epoch 1:
Train Loss: 3.0669 | Train Acc@1: 11.11% | Train Acc@5: 52.69%
Val Loss: 2.8965 | Val Acc@1: 11.61% | Val Acc@5: 53.38%
Train–Test Gap: 0.50% | ECE: 0.0576 | MOC: 0.4943
⏱ Epoch Time: 14.93s


Epoch 2/100: 100%|██████████| 391/391 [00:13<00:00, 29.48it/s]



📊 Epoch 2:
Train Loss: 2.8134 | Train Acc@1: 11.63% | Train Acc@5: 54.44%
Val Loss: 2.3706 | Val Acc@1: 10.00% | Val Acc@5: 49.72%
Train–Test Gap: 1.63% | ECE: 0.0585 | MOC: 0.5568
⏱ Epoch Time: 15.78s


Epoch 3/100: 100%|██████████| 391/391 [00:12<00:00, 31.41it/s]


...


📊 Epoch 38:
Train Loss: 1.0735 | Train Acc@1: 61.77% | Train Acc@5: 96.22%
Val Loss: 1.0128 | Val Acc@1: 64.30% | Val Acc@5: 96.61%
Train–Test Gap: 2.53% | ECE: 0.0352 | MOC: 0.3590
⏱ Epoch Time: 14.07s


Epoch 39/100: 100%|██████████| 391/391 [00:11<00:00, 33.14it/s]



📊 Epoch 39:
Train Loss: 1.0603 | Train Acc@1: 61.94% | Train Acc@5: 96.40%
Val Loss: 1.0246 | Val Acc@1: 63.46% | Val Acc@5: 96.53%
Train–Test Gap: 1.52% | ECE: 0.0208 | MOC: 0.3525
⏱ Epoch Time: 13.92s


Epoch 40/100: 100%|██████████| 391/391 [00:12<00:00, 30.50it/s]



📊 Epoch 40:
Train Loss: 1.0549 | Train Acc@1: 62.49% | Train Acc@5: 96.51%
Val Loss: 1.1583 | Val Acc@1: 62.81% | Val Acc@5: 96.57%
Train–Test Gap: 0.32% | ECE: 0.0360 | MOC: 0.3486
⏱ Epoch Time: 15.49s

⏹ Early stopping triggered after 40 epochs (no improvement in 5 epochs).

✅ Training Complete.
Best Top-1 Accuracy: 64.49% at Epoch 35
Total Training Time: 9.53 minutes

⚡ Inference Time (1 sample): 49.356 ms
